In [1]:
import mne
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from moabb.datasets import Zhou2020
import plotly.io as pio

pio.renderers.default = "vscode"

In [ ]:
# SELECIONAR PARTICIPANTES,TRIAL,SESSÃO,CLASSE
# como o dataset de cada sujeito é separado é possível extrair mais levemente as informações

import os
os.environ["MNE_DATA"] = "/home/machlearn/mne_data"  # ajuste para o seu caminho

import mne
from moabb.datasets import Zhou2020

dataset = Zhou2020()
data = dataset.get_data()

participantes = list(data.keys())
participantes = sorted([p for p in participantes if 1 <= int(p) <= 12], key=int) # pega apenas os 12 primeiros participantes
cache_epochs = {}  # evita reprocessar mne.Epochs a cada troca

def carregar_epochs(participante, sessao):
    chave = (participante, sessao)
    if chave in cache_epochs:
        return cache_epochs[chave]

    run_key = list(data[participante][sessao].keys())[0]
    raw = data[participante][sessao][run_key]

    events, event_id = mne.events_from_annotations(raw)
    epochs = mne.Epochs(raw, events, event_id=event_id,
                         tmin=0, tmax=3.0, baseline=None, preload=True)
    epochs.pick('eeg')

    cache_epochs[chave] = epochs
    return epochs

# --- dropdowns ---
dd_sujeito = widgets.Dropdown(options=participantes, description='Sujeito:')
dd_sessao  = widgets.Dropdown(description='Sessão:')
dd_classe  = widgets.Dropdown(description='Classe:')
dd_trial   = widgets.Dropdown(description='Trial:')

fig = go.FigureWidget()
fig.update_layout(
    xaxis_title="Tempo (s)",
    yaxis_title="Amplitude (µV)",
    legend_title="Canal",
    margin=dict(l=10, r=10, t=60, b=10),
    width=1300,   
    height=500 
)

ch_names_atual = []

def plotar():
    participante = dd_sujeito.value
    sessao = dd_sessao.value
    classe = dd_classe.value
    trial_num = dd_trial.value
    if None in (participante, sessao, classe, trial_num):
        return

    epochs = carregar_epochs(participante, sessao)
    ep_classe = epochs[classe]
    trial_data = ep_classe[trial_num - 1].get_data()[0]
    times = ep_classe.times

    global ch_names_atual
    with fig.batch_update():
        if ch_names_atual != epochs.ch_names:
            fig.data = []
            for ch in epochs.ch_names:
                fig.add_trace(go.Scatter(x=times, y=[], mode='lines', name=ch))
            ch_names_atual = epochs.ch_names[:]
        for i in range(len(ch_names_atual)):
            fig.data[i].x = times
            fig.data[i].y = trial_data[i]
        fig.layout.title = f"Sujeito {participante} | Sessão {sessao} | Classe: {classe} | Trial {trial_num}"

def atualizar_sessoes(change=None):
    sessoes = list(data[dd_sujeito.value].keys())
    dd_sessao.options = sessoes
    if sessoes:
        dd_sessao.value = sessoes[0]   # <-- força a seleção, não confia no auto-select
    atualizar_classes()

def atualizar_classes(change=None):
    participante, sessao = dd_sujeito.value, dd_sessao.value
    if participante is None or sessao is None:
        return
    epochs = carregar_epochs(participante, sessao)
    classes = list(epochs.event_id.keys())
    dd_classe.options = classes
    if classes:
        dd_classe.value = classes[0]   # <-- idem
    atualizar_trials()

def atualizar_trials(change=None):
    participante, sessao, classe = dd_sujeito.value, dd_sessao.value, dd_classe.value
    if None in (participante, sessao, classe):
        return
    epochs = carregar_epochs(participante, sessao)
    n_trials = len(epochs[classe])
    trials = list(range(1, n_trials + 1))
    dd_trial.options = trials
    if trials:
        dd_trial.value = trials[0]     # <-- idem
    plotar()

dd_sujeito.observe(atualizar_sessoes, names='value')
dd_sessao.observe(atualizar_classes, names='value')
dd_classe.observe(atualizar_trials, names='value')
dd_trial.observe(lambda change: plotar(), names='value')

atualizar_sessoes()

controles = widgets.HBox([dd_sujeito, dd_sessao, dd_classe, dd_trial])
display(controles, fig)

FileNotFoundError: Download location /media/machlearn/B88CDDAF8CDD6900 as specified by MNE_DATA does not exist. Either create this directory manually and try again, or set MNE_DATA to an existing directory.